# Sentinel-3 OLCI Processing Pipeline (CPR ROI)

**Stage 1 of 2.** Downloads OLCI L2 WFR scenes for every CPR sample date, applies the water-quality flag mask, and extracts a 4 km box around each CPR point. Stage 2 (`DE2000_CPR_Matchup.ipynb`) does the eRGB conversion, DE2000 anomaly detection, and the CPR comparison.

## Overview

This notebook carries out the satellite data acquisition and cropping described in Section 2.2 of the dissertation. For every CPR sample date, it:

1. Searches the Copernicus Data Space catalogue for every Sentinel-3 OLCI scene intersecting that date's CPR points, using a bounding box built from the points themselves (Section 2.2).
2. Downloads each scene, applies the WQSF quality flag mask (Appendix Table A1), and reads all twelve OLCI visible/NIR reflectance bands, not only the three used for eRGB, so the quality checks in Stage 2 have the full spectrum to work with.
3. For every CPR point the scene covers, extracts the swath pixels within a 4 km box centred on that point, with the box size converted from kilometres to degrees at that point's own latitude (Section 2.2), and saves both the raw swath pixels and a regularly gridded version for display.
4. Deletes the downloaded scene immediately after its boxes are cut, so disk usage stays roughly constant regardless of how many scenes are processed.

No time filtering happens in this notebook. Every scene found on a CPR sample date is cropped and its acquisition timestamp is stored in the output filename, so the matchup time window can be chosen and varied freely in Stage 2 without re-running any of this.

## Why the box is built per point rather than per scene

A regular latitude/longitude grid is only locally correct: a degree of longitude covers less ground distance than a degree of latitude, and by how much depends on latitude itself (at 53°N, about 60% as much). A single grid built for an entire scene can therefore only be correct along one row of it. This pipeline instead builds a separate small grid for each CPR point, sized from that point's own latitude, so grid cells cover the same ground distance everywhere.

The per-point grid is used only for the saved display image. OLCI's pixel footprint is about 300 m at nadir but grows to roughly 780 m at the swath edge, so a fixed square grid cannot evenly represent it everywhere in the swath; nearest-neighbour resampling is used instead of binning, which avoids leaving cells empty purely from a mismatch between the grid and the pixel layout, while still leaving genuinely cloud- or land-masked cells as no-data. Every statistic used in Stage 2 is computed from the raw swath pixels saved alongside the display grid, not from this resampled version, so nothing in the reported results depends on this resampling step at all.

## Settings

Fixed paths, search parameters, and the OLCI band and quality-flag lists used throughout (Section 2.2). The crop box is 4 km square, matching the pixel-averaging area used for the equivalent matchup in Shunmugapandi et al. (2026). `LAND` is included in the applied flag list, alongside the standard cloud, quality, and atmospheric-correction failure flags recommended in the EUMETSAT product notice (Appendix Table A1).

In [ ]:
import os
import re
import glob
import json
import time
import shutil
import zipfile
import getpass
from datetime import datetime

import numpy as np
import pandas as pd
import h5py
import requests
from scipy.spatial import cKDTree   # nearest-neighbour resampling for the display grid

# ============================================================
# ROOT FOLDER
# Must NOT be under iCloud Drive, OneDrive, or any synced folder.
# ============================================================

ROOT = os.path.expanduser('~/CPR_ROI_Sentinel3')

CPR_ROI_FILE = os.path.join(ROOT, 'CPR_ROI.xlsx')

# --- Working folders ---
# L2 and L2_masked only ever hold ONE scene at a time (deleted immediately after
# that scene's boxes are cut), so they stay small. There is no longer a
# full-scene reprojected file at all.
DOWNLOAD_FOLDER = os.path.join(ROOT, 'Data', 'L2')
MASKED_FOLDER   = os.path.join(ROOT, 'Data', 'L2_masked')
CROPPED_FOLDER  = os.path.join(ROOT, 'Data', 'L2_reprojected', 'L2_cropped')
STATUS_FOLDER   = os.path.join(ROOT, 'Status')

for folder in [DOWNLOAD_FOLDER, MASKED_FOLDER, CROPPED_FOLDER, STATUS_FOLDER]:
    os.makedirs(folder, exist_ok=True)

STATUS_LOG_FILE       = os.path.join(STATUS_FOLDER, 'cpr_pipeline_status.csv')
SCENE_CHECKPOINT_FILE = os.path.join(STATUS_FOLDER, 'cpr_scene_checkpoint.json')

# --- Search / download / disk safety ---
SEARCH_BUFFER_DEG = 0.25   # padding around each date-group bounding box, for the search only
MAX_RETRIES       = 5
BACKOFF_BASE_SEC  = 5
MIN_FREE_GB       = 5.0

# --- Crop box: 4 km square, converted to degrees at each point's own latitude ---
# Matches the 4 x 4 MODIS pixel (approx 4 km) averaging area used for the
# pixel-based matchup in Shunmugapandi et al. (2026).
BOX_WIDTH_KM   = 4.0
KM_PER_DEG_LAT = 111.32
RESOLUTION     = 0.003   # degrees of latitude (approx 300 m), display grid spacing


def box_half_width_deg(lat_deg):
    """(lat_buffer_deg, lon_buffer_deg) for a BOX_WIDTH_KM square centred at lat_deg."""
    half_km = BOX_WIDTH_KM / 2
    lat_buf = half_km / KM_PER_DEG_LAT
    lon_buf = half_km / (KM_PER_DEG_LAT * np.cos(np.radians(lat_deg)))
    return lat_buf, lon_buf


# --- OLCI bands ---
# Oa01-Oa12 cover 400-754 nm. Oa03/Oa04/Oa06 are the eRGB bands (the OLCI
# equivalents of MODIS 443/488/555). The rest are carried so the negative-Rrs
# quality check used by McCarry (2023) and Shunmugapandi (2025) can be run
# across the full spectrum, not just the three eRGB bands.
BAND_INFO = [
    ('Oa01_reflectance.nc', 'Rrs_400', 400.00),
    ('Oa02_reflectance.nc', 'Rrs_412', 412.50),
    ('Oa03_reflectance.nc', 'Rrs_442', 442.50),   # eRGB blue
    ('Oa04_reflectance.nc', 'Rrs_490', 490.00),   # eRGB green
    ('Oa05_reflectance.nc', 'Rrs_510', 510.00),
    ('Oa06_reflectance.nc', 'Rrs_560', 560.00),   # eRGB red
    ('Oa07_reflectance.nc', 'Rrs_620', 620.00),
    ('Oa08_reflectance.nc', 'Rrs_665', 665.00),
    ('Oa09_reflectance.nc', 'Rrs_674', 673.75),
    ('Oa10_reflectance.nc', 'Rrs_681', 681.25),
    ('Oa11_reflectance.nc', 'Rrs_709', 708.75),
    ('Oa12_reflectance.nc', 'Rrs_754', 753.75),
]
BAND_FILES    = [b[0] for b in BAND_INFO]
BAND_NAME_MAP = {b[0]: b[1] for b in BAND_INFO}
BAND_WAVELENGTHS = {b[1]: b[2] for b in BAND_INFO}

ERGB_BANDS = ('Rrs_442', 'Rrs_490', 'Rrs_560')   # blue, green, red

# WQSF flags to mask, following the EUMETSAT product notice (Appendix Table A1).
# The land flag is ALSO saved separately so figures can shade land distinctly
# from cloud.
RECOM_FLAG_NAMES = [
    'LAND',
    'CLOUD', 'CLOUD_AMBIGUOUS', 'CLOUD_MARGIN', 'INVALID',
    'COSMETIC', 'SATURATED', 'SUSPECT', 'HISOLZEN',
    'HIGHGLINT', 'SNOW_ICE', 'AC_FAIL', 'WHITECAPS', 'ADJAC',
    'RWNEG_O2', 'RWNEG_O3', 'RWNEG_O4', 'RWNEG_O5',
    'RWNEG_O6', 'RWNEG_O7', 'RWNEG_O8',
]

print(f'Root folder: {ROOT}')
print(f'{len(BAND_FILES)} OLCI bands will be saved per box '
      f'(eRGB uses {", ".join(ERGB_BANDS)})')
print('All working folders created (or already existed).')

## Disk space check

Called before every scene download. Stops the run cleanly rather than filling the disk. Because each scene is deleted as soon as its boxes are cut, peak usage stays roughly constant, about one scene's worth of disk space, regardless of how many scenes are processed in total.

In [ ]:
def free_space_gb(path):
    total, used, free = shutil.disk_usage(path)
    return free / (1024 ** 3)


def check_disk_space(min_free_gb=MIN_FREE_GB):
    free_gb = free_space_gb(ROOT)
    if free_gb < min_free_gb:
        raise RuntimeError(
            f'Only {free_gb:.1f} GB free, below the {min_free_gb} GB safety threshold. '
            'Stopping before starting a new download. Free up space and re-run this cell, '
            'it will pick up where it left off.'
        )
    return free_gb


print(f'Current free space: {free_space_gb(ROOT):.1f} GB')

## Credentials

Prompts for both the Copernicus Data Space username and password at the start of each session, rather than storing either in the notebook.

In [ ]:
CDSE_USERNAME = input('Copernicus Data Space username (email): ')
CDSE_PASSWORD = getpass.getpass('Copernicus Data Space password: ')

TOKEN_URL = 'https://identity.dataspace.copernicus.eu/auth/realms/CDSE/protocol/openid-connect/token'


def get_token(username, password, max_retries=MAX_RETRIES):
    last_exc = None
    for attempt in range(1, max_retries + 1):
        try:
            resp = requests.post(
                TOKEN_URL,
                data={
                    'grant_type': 'password',
                    'username':   username,
                    'password':   password,
                    'client_id':  'cdse-public',
                },
                timeout=30,
            )
            resp.raise_for_status()
            return resp.json()['access_token']
        except requests.exceptions.RequestException as exc:
            last_exc = exc
            wait = BACKOFF_BASE_SEC * (2 ** (attempt - 1))
            print(f'  Token request failed (attempt {attempt}/{max_retries}): {exc}')
            if attempt < max_retries:
                print(f'  Retrying in {wait}s...')
                time.sleep(wait)
    raise last_exc


try:
    _tok = get_token(CDSE_USERNAME, CDSE_PASSWORD)
    print('Connected to Copernicus Data Space successfully.')
except Exception as e:
    print(f'Authentication failed after {MAX_RETRIES} attempts: {e}')

## Load CPR ROI data and build date groups

Loads `CPR_ROI.xlsx` and groups its points by sample date, since points sampled on the same day are searched together using one combined bounding box (Section 2.2). `roi_row_id` is the row index of `CPR_ROI.xlsx` and is stored in every cropped file, so Stage 2 can join back to the in situ AEI values. The matchup notebook rebuilds this index the same way, so the two notebooks must not sort or filter `df_roi` differently.

In [ ]:
df_roi = pd.read_excel(CPR_ROI_FILE)
df_roi['datetime'] = pd.to_datetime(df_roi['midpoint_date_gmt'], utc=True)
df_roi['date'] = df_roi['datetime'].dt.date
df_roi = df_roi.reset_index(drop=True)
df_roi['roi_row_id'] = df_roi.index

print(f'Loaded {len(df_roi)} CPR points across {df_roi["date"].nunique()} unique dates')

date_groups = []
for d, group in df_roi.groupby('date'):
    west  = group['longitude'].min() - SEARCH_BUFFER_DEG
    east  = group['longitude'].max() + SEARCH_BUFFER_DEG
    south = group['latitude'].min()  - SEARCH_BUFFER_DEG
    north = group['latitude'].max()  + SEARCH_BUFFER_DEG
    date_groups.append({
        'date': d,
        'west': west, 'east': east, 'south': south, 'north': north,
        'rows': group[['roi_row_id', 'latitude', 'longitude']].to_dict('records'),
    })

print(f'Built {len(date_groups)} date groups')

# Quick reality check on how many points can possibly get a same-day matchup.
# OLCI is a passive optical sensor with a single daylight pass per day here
# (roughly 10:00-11:30 UTC at these longitudes), so a CPR tow at 03:00 will
# never be within a few hours of an overpass no matter how many scenes exist.
_h = df_roi['datetime'].dt.hour + df_roi['datetime'].dt.minute / 60
for _w in (4, 6, 8, 12):
    _n = int((np.minimum(abs(_h - 10.75), 24 - abs(_h - 10.75)) <= _w).sum())
    print(f'  {_n:5d} / {len(df_roi)} points ({100 * _n / len(df_roi):.0f}%) are within '
          f'+/-{_w}h of a nominal 10:45 UTC overpass (before any cloud loss)')

## Status log helper

Appends one row per CPR point per processing stage (search, download, crop) to a CSV log, so every point's outcome, including failures, is recorded even if the run is interrupted and restarted partway through.

In [ ]:
def log_status(roi_row_id, date, lat, lon, stage, status, detail=''):
    row = pd.DataFrame([{
        'roi_row_id': roi_row_id,
        'date': str(date),
        'latitude': lat,
        'longitude': lon,
        'stage': stage,
        'status': status,
        'detail': detail,
        'timestamp': datetime.utcnow().isoformat(timespec='seconds'),
    }])
    write_header = not os.path.exists(STATUS_LOG_FILE)
    row.to_csv(STATUS_LOG_FILE, mode='a', header=write_header, index=False)


print(f'Status log will be written to: {STATUS_LOG_FILE}')

## Search, download, mask and crop helper functions

Defines the functions that do the main work: searching the Copernicus catalogue for a date's scenes, downloading a scene, reading its quality flags and reflectance bands, and cropping a box for each CPR point the scene covers.

Reading is done in two passes to keep memory bounded: geolocation and flags first, to work out which swath pixels any point actually needs, then one band file at a time, keeping only those pixels. Quality flags and reflectance bands are read by variable name rather than by position in the file, so the correct variable is always used even if a product's internal ordering differs.

In [ ]:
CATALOGUE_URL = 'https://catalogue.dataspace.copernicus.eu/odata/v1/Products'
DOWNLOAD_URL  = 'https://download.dataspace.copernicus.eu/odata/v1/Products({id})/$value'


def load_checkpoint():
    if os.path.exists(SCENE_CHECKPOINT_FILE):
        with open(SCENE_CHECKPOINT_FILE) as f:
            return set(json.load(f))
    return set()


def save_checkpoint(done_scenes):
    with open(SCENE_CHECKPOINT_FILE, 'w') as f:
        json.dump(sorted(done_scenes), f)


def search_scenes(roi_date, west, south, east, north, max_retries=MAX_RETRIES):
    wkt = (f'POLYGON(({west} {south},{east} {south},'
           f'{east} {north},{west} {north},{west} {south}))')
    start_str = f'{roi_date}T00:00:00.000Z'
    end_str   = f'{roi_date}T23:59:59.000Z'
    filter_str = (
        "Collection/Name eq 'SENTINEL-3' "
        "and Attributes/OData.CSC.StringAttribute/any("
        "    att:att/Name eq 'productType' "
        "    and att/OData.CSC.StringAttribute/Value eq 'OL_2_WFR___') "
        f'and ContentDate/Start gt {start_str} '
        f'and ContentDate/Start lt {end_str} '
        f"and OData.CSC.Intersects(area=geography'SRID=4326;{wkt}')"
    )
    last_exc = None
    for attempt in range(1, max_retries + 1):
        try:
            resp = requests.get(
                CATALOGUE_URL,
                params={'$filter': filter_str, '$orderby': 'ContentDate/Start', '$top': 20},
                timeout=60,
            )
            resp.raise_for_status()
            return resp.json().get('value', [])
        except requests.exceptions.RequestException as exc:
            last_exc = exc
            wait = BACKOFF_BASE_SEC * (2 ** (attempt - 1))
            print(f'    Search failed (attempt {attempt}/{max_retries}): {exc}')
            if attempt < max_retries:
                time.sleep(wait)
    raise last_exc


def download_scene(product_id, product_name, folder, username, password, max_retries=MAX_RETRIES):
    sen3_path = os.path.join(folder, product_name + '.SEN3')
    if os.path.exists(sen3_path):
        return True

    check_disk_space()
    zip_path = os.path.join(folder, product_name + '.zip')
    last_exc = None

    for attempt in range(1, max_retries + 1):
        try:
            token   = get_token(username, password)
            headers = {'Authorization': f'Bearer {token}'}
            url     = DOWNLOAD_URL.format(id=product_id)

            print(f'    Downloading, attempt {attempt}/{max_retries}...')
            with requests.get(url, headers=headers, stream=True,
                              allow_redirects=True, timeout=300) as r:
                r.raise_for_status()
                total = int(r.headers.get('content-length', 0))
                downloaded = 0
                with open(zip_path, 'wb') as f:
                    for chunk in r.iter_content(chunk_size=1024 * 1024):
                        f.write(chunk)
                        downloaded += len(chunk)
                        if total:
                            print(f'      {downloaded/1e6:.0f} / {total/1e6:.0f} MB', end='\r')
            print()

            with zipfile.ZipFile(zip_path, 'r') as z:
                z.extractall(folder)
            os.remove(zip_path)
            return True

        except (requests.exceptions.RequestException, zipfile.BadZipFile, OSError) as exc:
            last_exc = exc
            if os.path.exists(zip_path):
                os.remove(zip_path)
            wait = BACKOFF_BASE_SEC * (2 ** (attempt - 1))
            print(f'    Download failed (attempt {attempt}/{max_retries}): {exc}')
            if attempt < max_retries:
                time.sleep(wait)

    print(f'    Giving up after {max_retries} attempts: {last_exc}')
    return False


# ------------------------------------------------------------------
# Reading OLCI files
# ------------------------------------------------------------------

def _open_2d_named(filepath, wanted_names):
    """Return (data, attrs_dict) for the first variable in `filepath` whose name matches
    one of `wanted_names` (case-insensitive). Falls back to the first 2D variable only if
    no name matches, and says so, rather than silently guessing."""
    try:
        with h5py.File(filepath, 'r') as f:
            keys = list(f.keys())
            match = next((k for k in keys if k.lower() in wanted_names), None)
            if match is None:
                match = next((k for k in keys if f[k].ndim == 2), None)
                if match is None:
                    raise ValueError(f'No 2D variable found in {filepath}')
                print(f'      note: no variable named {wanted_names} in '
                      f'{os.path.basename(filepath)}, using {match}')
            var = f[match]
            return var[:], dict(var.attrs)
    except OSError:
        import netCDF4 as nc
        with nc.Dataset(filepath, 'r') as ds:
            keys = list(ds.variables)
            match = next((k for k in keys if k.lower() in wanted_names), None)
            if match is None:
                match = next((k for k in keys if ds.variables[k].ndim == 2), None)
                if match is None:
                    raise ValueError(f'No 2D variable found in {filepath}')
            var = ds.variables[match]
            attrs = {a: getattr(var, a) for a in var.ncattrs()}
            return np.array(var[:]), attrs


def read_wqsf(scene_path):
    """Read the water-quality flags and their bit meanings from wqsf.nc, by name."""
    filepath = os.path.join(scene_path, 'wqsf.nc')
    data, attrs = _open_2d_named(filepath, {'wqsf'})
    wqsf = np.asarray(data).astype(np.uint64)

    flag_meanings = attrs.get('flag_meanings')
    flag_masks    = attrs.get('flag_masks')
    if flag_meanings is None or flag_masks is None:
        raise ValueError(f'wqsf.nc in {scene_path} has no flag_meanings/flag_masks attributes')
    if isinstance(flag_meanings, bytes):
        flag_meanings = flag_meanings.decode('utf-8')
    names = str(flag_meanings).split()
    flag_bit_map = {n: int(m) for n, m in zip(names, np.asarray(flag_masks).ravel())}
    return wqsf, flag_bit_map


def build_flag_mask(wqsf, flag_bit_map, flag_names_to_apply):
    combined = np.zeros(wqsf.shape, dtype=bool)
    missing = []
    for flag_name in flag_names_to_apply:
        if flag_name not in flag_bit_map:
            missing.append(flag_name)
            continue
        combined |= (wqsf & np.uint64(flag_bit_map[flag_name])) != 0
    return combined, missing


def read_geo(scene_path):
    filepath = os.path.join(scene_path, 'geo_coordinates.nc')
    try:
        with h5py.File(filepath, 'r') as f:
            lat = f['latitude'][:].astype(np.float64) * 1e-6
            lon = f['longitude'][:].astype(np.float64) * 1e-6
    except OSError:
        import netCDF4 as nc
        with nc.Dataset(filepath, 'r') as ds:
            lat = np.array(ds.variables['latitude'][:], dtype=np.float64) * 1e-6
            lon = np.array(ds.variables['longitude'][:], dtype=np.float64) * 1e-6
    return lat, lon


def read_reflectance_band(filepath):
    """Read one OLCI reflectance band and convert to Rrs (sr^-1).

    The Oa*_reflectance products are normalised water-leaving REFLECTANCE, so
    dividing by pi gives remote sensing reflectance, which is what the LUTs and
    the eRGB stretch maxima are defined in.
    Fill values are removed BEFORE scale/offset are applied.
    """
    var_stem = os.path.basename(filepath).replace('.nc', '')
    try:
        with h5py.File(filepath, 'r') as f:
            keys = list(f.keys())
            key = next((k for k in keys if k == var_stem), None)
            if key is None:
                key = [k for k in keys
                       if 'reflectance' in k and 'unc' not in k and 'err' not in k][0]
            var    = f[key]
            raw    = var[:].astype(np.float64)
            scale  = np.ravel(var.attrs.get('scale_factor', [1.0]))[0]
            offset = np.ravel(var.attrs.get('add_offset',   [0.0]))[0]
            fill   = np.ravel(var.attrs.get('_FillValue',  [65535]))[0]
    except OSError:
        import netCDF4 as nc
        with nc.Dataset(filepath, 'r') as ds:
            keys = list(ds.variables)
            key = next((k for k in keys if k == var_stem), None)
            if key is None:
                key = [k for k in keys if 'reflectance' in k and 'unc' not in k][0]
            var    = ds.variables[key]
            raw    = np.array(var[:], dtype=np.float64)
            scale  = float(getattr(var, 'scale_factor', 1.0))
            offset = float(getattr(var, 'add_offset', 0.0))
            fill   = float(getattr(var, '_FillValue', 65535))

    raw[raw == fill] = np.nan
    return (raw * scale + offset) / np.pi


# ------------------------------------------------------------------
# Local per-point gridding
# ------------------------------------------------------------------

def build_local_grid(roi_lat, roi_lon):
    """Regular grid for ONE box, using THIS point's latitude for the longitude spacing
    so that lat and lon cells cover the same ground distance (approx RESOLUTION degrees
    of latitude, about 300 m). Returns edges and centres for both axes."""
    lat_buf, lon_buf = box_half_width_deg(roi_lat)
    res_lat = RESOLUTION
    res_lon = RESOLUTION / np.cos(np.radians(roi_lat))

    n_lat = max(int(round(2 * lat_buf / res_lat)), 1)
    n_lon = max(int(round(2 * lon_buf / res_lon)), 1)

    lat_edges = (roi_lat - lat_buf) + np.arange(n_lat + 1) * res_lat
    lon_edges = (roi_lon - lon_buf) + np.arange(n_lon + 1) * res_lon
    lat_centres = 0.5 * (lat_edges[:-1] + lat_edges[1:])
    lon_centres = 0.5 * (lon_edges[:-1] + lon_edges[1:])
    return lat_edges, lon_edges, lat_centres, lon_centres, res_lat, res_lon


def grid_box(swath_lat, swath_lon, values, land_flag, roi_lat, roi_lon,
             max_cells_away=1.5):
    """Nearest-neighbour resample the swath pixels onto the local display grid.

    Why nearest neighbour rather than binning into cells:

    OLCI's across-track pixel size is about 300 m at nadir but grows to roughly 780 m at
    the swath edge, while along-track stays near 300 m. The pixel lattice is therefore
    strongly anisotropic, and its spacing depends on where in the swath the box happens
    to fall. Binning onto any fixed square grid leaves whole columns of cells with no
    contributing pixel whenever the box sits away from nadir. Tested on a simulated swath,
    a 300 m grid is 0.7% empty at nadir but 57% empty at the swath edge, in exactly the
    column-stripe pattern seen in the earlier outputs. Making the cell size adapt to the
    pixel count helps but cannot fix it, because a square grid cannot match an anisotropic
    lattice: it still leaves about a third of cells empty at the edge.

    Nearest-neighbour assignment removes the problem entirely. Every cell takes the value
    of the closest swath pixel, so aliasing gaps disappear while genuine gaps survive:
    cloud- and land-masked pixels are still present in the swath arrays carrying NaN, so
    the nearest pixel to a clouded cell is itself NaN and the hole is preserved. Cells
    further than `max_cells_away` cells from any swath pixel are left NaN, which only
    happens at the very edge of the box.

    This is a DISPLAY grid only. Every number in the results comes from the raw swath
    pixels, which are saved alongside it and are untouched by any resampling.

    Returns (grids dict, nn_distance_km, grid_land, lat_centres, lon_centres).
    """
    from scipy.spatial import cKDTree

    lat_edges, lon_edges, lat_c, lon_c, res_lat, res_lon = build_local_grid(roi_lat, roi_lon)
    n_lat, n_lon = len(lat_c), len(lon_c)

    if swath_lat.size == 0:
        nan_grid = np.full((n_lat, n_lon), np.nan, dtype=np.float32)
        return ({k: nan_grid.copy() for k in values}, nan_grid.copy(),
                np.zeros((n_lat, n_lon), bool), lat_c, lon_c)

    # Work in km so that latitude and longitude distances are directly comparable.
    coslat = np.cos(np.radians(roi_lat))
    tree = cKDTree(np.column_stack([swath_lat * KM_PER_DEG_LAT,
                                    swath_lon * KM_PER_DEG_LAT * coslat]))

    lon_mesh, lat_mesh = np.meshgrid(lon_c, lat_c)
    dist_km, idx = tree.query(np.column_stack([lat_mesh.ravel() * KM_PER_DEG_LAT,
                                               lon_mesh.ravel() * KM_PER_DEG_LAT * coslat]),
                              k=1)

    cell_km = res_lat * KM_PER_DEG_LAT
    too_far = dist_km > max_cells_away * cell_km

    grids = {}
    for band_name, vals in values.items():
        g = vals[idx].astype(np.float64)
        g[too_far] = np.nan
        grids[band_name] = g.reshape(n_lat, n_lon).astype(np.float32)

    grid_land = land_flag[idx].copy()
    grid_land[too_far] = False
    grid_land = grid_land.reshape(n_lat, n_lon)

    nn_distance = dist_km.reshape(n_lat, n_lon).astype(np.float32)
    return grids, nn_distance, grid_land, lat_c, lon_c


def crop_points_from_scene(scene_path, rows, roi_date, scene_name, out_folder):
    """Mask one scene and cut a box for every CPR point in `rows` that it covers.

    Two-pass read to keep memory bounded: geolocation and flags first (to find which
    swath pixels are needed at all), then one band file at a time.

    Returns (outcomes, scene_valid_pct) where outcomes maps roi_row_id -> status string.
    """
    outcomes = {}

    wqsf, flag_bit_map = read_wqsf(scene_path)
    combined_mask, missing = build_flag_mask(wqsf, flag_bit_map, RECOM_FLAG_NAMES)
    if missing:
        print(f'    note: flags not present in this product and therefore not applied: '
              f'{", ".join(missing)}')
    land_bit  = flag_bit_map.get('LAND')
    land_mask = ((wqsf & np.uint64(land_bit)) != 0) if land_bit else np.zeros(wqsf.shape, bool)
    del wqsf

    scene_valid_pct = 100.0 * (combined_mask.size - int(combined_mask.sum())) / combined_mask.size

    lat, lon = read_geo(scene_path)
    if lat.shape != combined_mask.shape:
        raise ValueError(f'geo grid {lat.shape} does not match wqsf grid {combined_mask.shape}')

    # --- pass 1: which swath pixels does any box need? ---
    per_point_idx = {}
    needed = []
    for row in rows:
        lat_buf, lon_buf = box_half_width_deg(row['latitude'])
        sel = ((lat >= row['latitude'] - lat_buf) & (lat <= row['latitude'] + lat_buf) &
               (lon >= row['longitude'] - lon_buf) & (lon <= row['longitude'] + lon_buf))
        idx = np.flatnonzero(sel.ravel())
        if idx.size == 0:
            outcomes[row['roi_row_id']] = 'outside_scene_extent'
            continue
        per_point_idx[row['roi_row_id']] = idx
        needed.append(idx)

    if not needed:
        return outcomes, scene_valid_pct

    union_idx = np.unique(np.concatenate(needed))
    pos_in_union = {rid: np.searchsorted(union_idx, idx) for rid, idx in per_point_idx.items()}

    lat_u  = lat.ravel()[union_idx]
    lon_u  = lon.ravel()[union_idx]
    land_u = land_mask.ravel()[union_idx]
    mask_u = combined_mask.ravel()[union_idx]
    del lat, lon, land_mask, combined_mask

    # --- pass 2: one band at a time, keeping only the pixels we need ---
    band_u = {}
    for band_file in BAND_FILES:
        band_path = os.path.join(scene_path, band_file)
        band_name = BAND_NAME_MAP[band_file]
        if not os.path.exists(band_path):
            print(f'    note: {band_file} missing from this product, skipping that band')
            continue
        full = read_reflectance_band(band_path)
        vals = full.ravel()[union_idx]
        del full
        vals[mask_u] = np.nan          # apply the WQSF mask, LAND included
        band_u[band_name] = vals

    if not all(b in band_u for b in ERGB_BANDS):
        for rid in per_point_idx:
            outcomes[rid] = 'no_ergb_bands'
        return outcomes, scene_valid_pct

    # --- build and save one file per point ---
    dt_m = re.search(r'(\d{8})T(\d{6})', scene_name)
    hhmm = dt_m.group(2)[:4] if dt_m else 'XXXX'

    row_lookup = {r['roi_row_id']: r for r in rows}
    for rid, pos in pos_in_union.items():
        row = row_lookup[rid]
        s_lat, s_lon = lat_u[pos], lon_u[pos]
        s_land = land_u[pos]
        s_vals = {name: vals[pos] for name, vals in band_u.items()}

        ergb_stack = np.stack([s_vals[b] for b in ERGB_BANDS], axis=0)
        n_valid_swath = int(np.sum(np.all(np.isfinite(ergb_stack), axis=0)))

        grids, nn_dist, grid_land, lat_c, lon_c = grid_box(
            s_lat, s_lon, s_vals, s_land, row['latitude'], row['longitude'])

        out_name = (f'{roi_date}_row{rid}_{row["latitude"]:.3f}N_'
                    f'{row["longitude"]:.3f}E_{hhmm}_cropped.h5')
        out_path = os.path.join(out_folder, out_name)

        with h5py.File(out_path, 'w') as out_f:
            # display grid
            out_f.create_dataset('latitude',  data=lat_c)
            out_f.create_dataset('longitude', data=lon_c)
            out_f.create_dataset('land_mask', data=grid_land, compression='gzip')
            out_f.create_dataset('nn_distance_km', data=nn_dist, compression='gzip')
            for name, g in grids.items():
                out_f.create_dataset(name, data=g, compression='gzip')

            # raw swath pixels -- the statistics in stage 2 use THESE, not the grid
            sw = out_f.create_group('swath')
            sw.create_dataset('latitude',  data=s_lat.astype(np.float32),  compression='gzip')
            sw.create_dataset('longitude', data=s_lon.astype(np.float32),  compression='gzip')
            sw.create_dataset('land',      data=s_land,                    compression='gzip')
            for name, v in s_vals.items():
                sw.create_dataset(name, data=v.astype(np.float32), compression='gzip')

            out_f.attrs['roi_row_id']       = rid
            out_f.attrs['roi_lat']          = row['latitude']
            out_f.attrs['roi_lon']          = row['longitude']
            out_f.attrs['box_width_km']     = BOX_WIDTH_KM
            out_f.attrs['scene_name']       = scene_name
            out_f.attrs['scene_date']       = str(roi_date)
            out_f.attrs['n_swath_pixels']   = int(s_lat.size)
            out_f.attrs['n_swath_valid']    = n_valid_swath
            out_f.attrs['grid_shape']       = np.array(nn_dist.shape)
            out_f.attrs['bands_saved']      = ','.join(sorted(s_vals))
            out_f.attrs['pipeline_version'] = 'v2_local_regrid'

        outcomes[rid] = 'saved' if n_valid_swath > 0 else 'saved_no_valid_pixels'

    return outcomes, scene_valid_pct


print('Helper functions defined.')

## Verification: one scene, one point

Run this before committing to the full loop. It downloads a single scene, cuts one box, and plots the eRGB.

What a correct result looks like:
* no empty columns or rows in a regular repeating pattern
* not a single flat block of uniform colour
* the nearest-pixel distance well under one cell width nearly everywhere, meaning the display grid is properly matched to the swath at this position. Any white in the eRGB should be real cloud or land, not a resampling gap
* the CPR point marked with a star sits inside the box

The eRGB stretch used here is the same standardised one used in Stage 2 (Section 2.3): fixed minimum of 0 and fixed maxima taken from the 90th percentile of each band in the Valente et al. (2022) global in situ Rrs dataset, with a gamma of 0.8 on blue. Nothing here is recalculated from the image itself.

In [ ]:
import matplotlib.pyplot as plt

# McCarry et al. (2023) standardised stretch: fixed maxima from the Valente et al. (2022)
# global in situ Rrs dataset (90th percentile of each band), gamma 0.8 on blue.
RED_MAX, GREEN_MAX, BLUE_MAX, GAMMA = 0.0077, 0.0071, 0.0095, 0.8


def quick_ergb(red, green, blue):
    """Standalone standardised eRGB, so this test does not depend on the matchup notebook."""
    raw_neg = (red < 0) | (green < 0) | (blue < 0)
    r = np.where(raw_neg, np.nan, red)   / RED_MAX
    g = np.where(raw_neg, np.nan, green) / GREEN_MAX
    b = np.where(raw_neg, np.nan, blue)  / BLUE_MAX
    rgb = np.stack([r, g, b], axis=-1).astype(np.float64)
    out = np.full_like(rgb, np.nan)
    valid   = np.all(np.isfinite(rgb), axis=-1)
    ok      = valid & np.all(rgb <= 1, axis=-1)
    over    = valid & ~ok
    out[ok] = rgb[ok]
    if np.any(over):
        pix  = rgb[over]
        pmin = pix.min(axis=-1, keepdims=True)
        pmax = pix.max(axis=-1, keepdims=True)
        den  = pmax - pmin
        den[den == 0] = 1.0
        out[over] = (pix - pmin) / den
    b_ch = out[:, :, 2]
    fin  = np.isfinite(b_ch)
    b_ch[fin] = b_ch[fin] ** GAMMA
    out[:, :, 2] = b_ch
    return out


KNOWN_GOOD_TEST_DATE = '2019-05-12'

test_group = next((g for g in date_groups if str(g['date']) == KNOWN_GOOD_TEST_DATE), None)
if test_group is None:
    print(f'{KNOWN_GOOD_TEST_DATE} not in this dataset, using the first date instead. '
          f'If that scene is heavily clouded just change KNOWN_GOOD_TEST_DATE and re-run.')
    test_group = date_groups[0]

print(f'Testing date {test_group["date"]} ({len(test_group["rows"])} CPR point(s))')
test_results = search_scenes(test_group['date'], test_group['west'], test_group['south'],
                             test_group['east'], test_group['north'])

if len(test_results) == 0:
    print('No scenes found for this date. Pick another KNOWN_GOOD_TEST_DATE and re-run.')
else:
    tp = test_results[0]
    test_pid, test_pname = tp['Id'], tp['Name'].replace('.SEN3', '')
    print(f'Testing scene: {test_pname}')
    test_sen3 = os.path.join(DOWNLOAD_FOLDER, test_pname + '.SEN3')

    if not download_scene(test_pid, test_pname, DOWNLOAD_FOLDER, CDSE_USERNAME, CDSE_PASSWORD):
        print('Download failed, cannot verify.')
    else:
        test_out = os.path.join(MASKED_FOLDER, 'verification_crops')
        os.makedirs(test_out, exist_ok=True)
        outcomes, valid_pct = crop_points_from_scene(
            test_sen3, test_group['rows'], test_group['date'], test_pname, test_out)
        print(f'Scene valid (unmasked) fraction: {valid_pct:.1f}%')
        print('Outcomes:', pd.Series(outcomes).value_counts().to_dict())

        saved = sorted(glob.glob(os.path.join(test_out, '*_cropped.h5')))
        best, best_frac = None, -1.0
        for fp in saved:
            with h5py.File(fp, 'r') as f:
                frac = f.attrs['n_swath_valid'] / max(f.attrs['n_swath_pixels'], 1)
            if frac > best_frac:
                best, best_frac = fp, frac

        if best is None:
            print('No boxes were saved for this scene, try a different date.')
        else:
            with h5py.File(best, 'r') as f:
                lat_c  = f['latitude'][:]
                lon_c  = f['longitude'][:]
                red    = f['Rrs_560'][:]
                green  = f['Rrs_490'][:]
                blue   = f['Rrs_442'][:]
                land   = f['land_mask'][:]
                nn_d   = f['nn_distance_km'][:]
                roi_la = f.attrs['roi_lat']
                roi_lo = f.attrs['roi_lon']
                n_sw   = f.attrs['n_swath_pixels']
                n_sv   = f.attrs['n_swath_valid']

            print(f'\nBox: grid {red.shape}, {n_sw} swath pixels, '
                  f'{n_sv} of them valid ({100*n_sv/max(n_sw,1):.0f}%)')
            print(f'nearest-pixel distance per cell (km): median {np.median(nn_d):.3f}, '
                  f'max {nn_d.max():.3f}')
            print('  (median well under 0.3 km means the display grid is well matched to the '
                  'swath. Any NaN in the eRGB below should be real cloud or land, not gaps.)')

            rgb = quick_ergb(red, green, blue)
            box_valid = float(np.mean(np.all(np.isfinite(rgb), axis=-1)))
            disp = np.nan_to_num(np.clip(rgb, 0, 1), nan=1.0)
            disp[land] = [0.55, 0.55, 0.55]

            extent = [lon_c.min(), lon_c.max(), lat_c.min(), lat_c.max()]
            fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))
            axes[0].imshow(disp, extent=extent, origin='lower', aspect='auto')
            axes[0].plot(roi_lo, roi_la, marker='*', markersize=18, markerfacecolor='white',
                         markeredgecolor='black', markeredgewidth=1.2, linestyle='none',
                         label='CPR sample location')
            axes[0].legend(loc='upper right', fontsize=8)
            axes[0].set_title(f'eRGB, {box_valid:.0%} valid\n(grey = land, white = cloud/no data)')
            axes[0].set_xlabel('Longitude'); axes[0].set_ylabel('Latitude')

            im = axes[1].imshow(nn_d, extent=extent, origin='lower', aspect='auto',
                                cmap='viridis')
            axes[1].set_title('distance to nearest swath pixel')
            axes[1].set_xlabel('Longitude')
            fig.colorbar(im, ax=axes[1], label='km')
            plt.tight_layout(); plt.show()

        shutil.rmtree(test_sen3, ignore_errors=True)
        shutil.rmtree(test_out, ignore_errors=True)
        print('\nTest files cleaned up. The full run below is unaffected by this test.')

## Main loop: one scene at a time, fully processed, then deleted

For each CPR date: search once using the union bounding box of that date's points, then for every scene found, download it, cut a box for each point the scene actually covers, log an outcome for every point, and delete the raw scene immediately. Scenes already completed in a previous run are skipped via the checkpoint file, so this cell is safe to interrupt and restart.

No time filtering happens here. Every scene on the date is cropped and its name, which carries the acquisition timestamp, is stored, so the matchup window is a free choice in Stage 2.

In [ ]:
done_scenes = load_checkpoint()
print(f'{len(done_scenes)} scene(s) already completed in a previous run.\n')

for group in date_groups:
    d = group['date']
    rows = group['rows']

    print(f'=== {d}  ({len(rows)} CPR point(s)) ===')

    try:
        results = search_scenes(d, group['west'], group['south'], group['east'], group['north'])
    except requests.exceptions.RequestException as exc:
        print(f'  Search permanently failed for {d}: {exc}')
        for row in rows:
            log_status(row['roi_row_id'], d, row['latitude'], row['longitude'],
                       stage='search', status='failed', detail=str(exc))
        continue

    if len(results) == 0:
        print('  No scenes found for this date')
        for row in rows:
            log_status(row['roi_row_id'], d, row['latitude'], row['longitude'],
                       stage='search', status='no_scene_found')
        continue

    for product in results:
        pid, pname = product['Id'], product['Name'].replace('.SEN3', '')

        if pname in done_scenes:
            print(f'  Already fully processed, skipping: {pname}')
            continue

        print(f'  --- Scene: {pname} ---')
        sen3_path = os.path.join(DOWNLOAD_FOLDER, pname + '.SEN3')

        if not download_scene(pid, pname, DOWNLOAD_FOLDER, CDSE_USERNAME, CDSE_PASSWORD):
            for row in rows:
                log_status(row['roi_row_id'], d, row['latitude'], row['longitude'],
                           stage='download', status='failed', detail=pname)
            continue

        try:
            outcomes, valid_pct = crop_points_from_scene(
                sen3_path, rows, d, pname, CROPPED_FOLDER)
            print(f'    Masked: {valid_pct:.1f}% of scene pixels valid')
        except Exception as exc:
            print(f'    Processing failed: {type(exc).__name__}: {exc}')
            for row in rows:
                log_status(row['roi_row_id'], d, row['latitude'], row['longitude'],
                           stage='crop', status='failed', detail=f'{pname}: {exc}')
            shutil.rmtree(sen3_path, ignore_errors=True)
            continue

        row_lookup = {r['roi_row_id']: r for r in rows}
        for rid, status in outcomes.items():
            r = row_lookup[rid]
            log_status(rid, d, r['latitude'], r['longitude'],
                       stage='crop', status=status, detail=pname)
        counts = pd.Series(outcomes).value_counts().to_dict() if outcomes else {}
        print(f'    Box outcomes: {counts}')

        shutil.rmtree(sen3_path, ignore_errors=True)
        done_scenes.add(pname)
        save_checkpoint(done_scenes)
        print(f'    Cleaned up. Free space now: {free_space_gb(ROOT):.1f} GB\n')

print('All dates processed.')

## Summary of what was produced

Run this to see how many points ended up with at least one cropped box, which is the ceiling on the final matchup count before any time-window or valid-pixel-fraction filtering is applied in Stage 2.

In [ ]:
cropped = sorted(glob.glob(os.path.join(CROPPED_FOLDER, '*_cropped.h5')))
print(f'{len(cropped)} cropped box file(s) on disk')

if cropped:
    recs = []
    for fp in cropped:
        with h5py.File(fp, 'r') as f:
            recs.append({
                'roi_row_id': int(f.attrs['roi_row_id']),
                'scene_date': str(f.attrs['scene_date']),
                'n_swath_pixels': int(f.attrs['n_swath_pixels']),
                'n_swath_valid':  int(f.attrs['n_swath_valid']),
            })
    summary = pd.DataFrame(recs)
    summary['valid_fraction'] = summary['n_swath_valid'] / summary['n_swath_pixels'].clip(lower=1)
    print(f'{summary["roi_row_id"].nunique()} of {len(df_roi)} CPR points have at least one box')
    print(f'Points with a box that is >=50% valid: '
          f'{summary[summary["valid_fraction"] >= 0.5]["roi_row_id"].nunique()}')
    print()
    print(summary['valid_fraction'].describe().to_string())

if os.path.exists(STATUS_LOG_FILE):
    log = pd.read_csv(STATUS_LOG_FILE)
    print('\nStatus log breakdown:')
    print(log.groupby(['stage', 'status']).size().to_string())

print('\nDone. Next: run DE2000_CPR_Matchup.ipynb.')

In [ ]:
import sys
print("Python:", sys.version)

import numpy, pandas, scipy, h5py
for pkg in (numpy, pandas, scipy, h5py):
    print(pkg.__name__, pkg.__version__)